# 📊 โปรเจกต์ ML: Data Preparation (การเตรียมข้อมูล)
**Dataset:** Student Spending Dataset  
**เป้าหมาย:** ทำความสะอาดข้อมูล จัดการความไม่สมบูรณ์ และเตรียมโครงสร้างให้พร้อมสำหรับการเทรนโมเดล Ensemble

## 1) Import Libraries (นำเข้าเครื่องมือที่จำเป็น)
โหลดไลบรารีพื้นฐานสำหรับการจัดการตารางข้อมูล (Pandas) และการบันทึกไฟล์ (Joblib)

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

## 2) Load Raw Data (โหลดข้อมูลดิบ)
นำเข้าข้อมูล `student_spending.csv` ที่โหลดมาจาก Kaggle เพื่อตรวจสอบหน้าตาข้อมูลเบื้องต้น

In [2]:
df = pd.read_csv('../data/student_spending.csv')
print(f"ขนาดข้อมูลดิบ: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์")
display(df.head())

ขนาดข้อมูลดิบ: 1000 แถว, 18 คอลัมน์


,Unnamed: 0,age,gender,year_in_school,major,monthly_income,financial_aid,tuition,housing,food,transportation,books_supplies,entertainment,personal_care,technology,health_wellness,miscellaneous,preferred_payment_method
0,0,19,Non-binary,Freshman,Psychology,958,270,5939,709,296,123,188,41,78,134,127,72,Credit/Debit Card
1,1,24,Female,Junior,Economics,1006,875,4908,557,365,85,252,74,92,226,129,68,Credit/Debit Card
2,2,24,Non-binary,Junior,Economics,734,928,3051,666,220,137,99,130,23,239,112,133,Cash
3,3,23,Female,Senior,Computer Science,617,265,4935,652,289,114,223,99,30,163,105,55,Mobile Payment App
4,4,20,Female,Senior,Computer Science,810,522,3887,825,372,168,194,48,71,88,71,104,Credit/Debit Card


## 3) Data Cleaning (การทำความสะอาดข้อมูลเพื่อลดความไม่สมบูรณ์)
ตรวจสอบและจัดการความไม่สมบูรณ์ของ Dataset (ตามข้อกำหนด 1.c):
- ลบคอลัมน์ `Unnamed: 0` ที่เป็นเพียง Index ขยะติดมาจากต้นฉบับ
- ตรวจสอบและจัดการ Missing Values

In [3]:
# ลบคอลัมน์ Unnamed: 0 (ถ้ามี)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print("✅ ลบคอลัมน์ 'Unnamed: 0' เรียบร้อยแล้ว")

# ตรวจสอบค่าว่าง
missing_val = df.isnull().sum().sum()
print(f"🔍 พบค่าว่างทั้งหมด: {missing_val} จุด")

if missing_val > 0:
    df = df.dropna()
    print("✅ ลบแถวที่มีค่าว่างทิ้งเรียบร้อยแล้ว")
else:
    print("✅ ข้อมูลชุดนี้ไม่มี Missing Values")

✅ ลบคอลัมน์ 'Unnamed: 0' เรียบร้อยแล้ว
🔍 พบค่าว่างทั้งหมด: 0 จุด
✅ ข้อมูลชุดนี้ไม่มี Missing Values


## 4) Encoding Categorical Data (แปลงข้อความเป็นตัวเลข)
โมเดลประมวลผลได้เฉพาะตัวเลข จึงต้องแปลงคอลัมน์ที่เป็นข้อความ (`gender`, `year_in_school`, `major`) ด้วยเทคนิค **One-Hot Encoding**

In [4]:
# คอลัมน์ที่ต้องแปลงเป็น 0 กับ 1 (เพิ่ม preferred_payment_method เข้าไปแล้ว)
categorical_cols = ['gender', 'year_in_school', 'major', 'preferred_payment_method']

# แปลงข้อความเป็นคอลัมน์ 0/1
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

print(f"✅ แปลงข้อมูลเสร็จสิ้น! จำนวนคอลัมน์ขยายเป็น: {df_encoded.shape[1]} คอลัมน์")
display(df_encoded.head())

✅ แปลงข้อมูลเสร็จสิ้น! จำนวนคอลัมน์ขยายเป็น: 28 คอลัมน์


,age,monthly_income,financial_aid,tuition,housing,food,transportation,books_supplies,entertainment,personal_care,...,year_in_school_Senior,year_in_school_Sophomore,major_Biology,major_Computer Science,major_Economics,major_Engineering,major_Psychology,preferred_payment_method_Cash,preferred_payment_method_Credit/Debit Card,preferred_payment_method_Mobile Payment App
0,19,958,270,5939,709,296,123,188,41,78,...,False,False,False,False,False,False,True,False,True,False
1,24,1006,875,4908,557,365,85,252,74,92,...,False,False,False,False,True,False,False,False,True,False
2,24,734,928,3051,666,220,137,99,130,23,...,False,False,False,False,True,False,False,True,False,False
3,23,617,265,4935,652,289,114,223,99,30,...,True,False,False,True,False,False,False,False,False,True
4,20,810,522,3887,825,372,168,194,48,71,...,True,False,False,True,False,False,False,False,True,False


## 5) Export Data & Features (บันทึกข้อมูลพร้อมใช้งาน)
บันทึกข้อมูลที่ผ่านการ Clean แล้วเพื่อนำไปเทรนโมเดล และบันทึก "รายชื่อคอลัมน์ (Features)" เป็นไฟล์ `.pkl` เพื่อนำไปใช้ตรวจสอบโครงสร้างข้อมูลในหน้า Web App

In [5]:
os.makedirs('../models', exist_ok=True)

# แยก Target ออกเพื่อเก็บเฉพาะรายชื่อ Features
X = df_encoded.drop(columns=['monthly_income'])
feature_names = X.columns.tolist()

# บันทึกไฟล์
joblib.dump(feature_names, '../models/ml_features.pkl')
df_encoded.to_csv('../data/student_spending_cleaned.csv', index=False)

print("💾 บันทึกโมเดล Features และข้อมูล Cleaned เรียบร้อย!")

💾 บันทึกโมเดล Features และข้อมูล Cleaned เรียบร้อย!
